# Assignment 2 - data checks and models

In [2]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import (GridSearchCV, StratifiedKFold,
                                     cross_validate, train_test_split)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

SEED = 0
SAMPLE_N = None  # None = all 257673 rows
NOMINAL = ["proto", "service", "state"]

CLASSES = (pd.read_csv("attack_category_map.csv")
             .sort_values("Mapping")["Attack Category Name"].tolist())

In [3]:
df = pd.read_csv("networkTraffic.csv", na_values=["?"])
df.shape

(257673, 44)

## Data checks

In [4]:
miss = df.isna().mean().mul(100).round(2)
miss[miss > 0]

service    54.85
dtype: float64

In [5]:
counts = df["attack_cat"].value_counts().sort_index()
print("imbalance ratio", round(counts.max() / counts.min(), 1))
pd.DataFrame({"class": CLASSES, "n": counts.values,
              "pct": (counts / len(df) * 100).round(3).values})

imbalance ratio 534.5


,class,n,pct
0,Normal,93000,36.092
1,Reconnaissance,13987,5.428
2,Backdoor,2329,0.904
3,DoS,16353,6.346
4,Exploits,44525,17.280
5,Analysis,2677,1.039
6,Fuzzers,24246,9.410
7,Worms,174,0.068
8,Shellcode,1511,0.586
9,Generic,58871,22.847


In [6]:
df[NOMINAL + ["id"]].nunique()

proto         133
service        12
state          11
id         257673
dtype: int64

In [7]:
num = df.drop(columns=["id", "attack_cat"] + NOMINAL)
num.agg(["min", "max", "std"]).T.sort_values("max", ascending=False).head(10)

,min,max,std
sload,0.0,5.988000e+09,1.857313e+08
stcpb,0.0,4.294959e+09,1.367795e+09
dtcpb,0.0,4.294882e+09,1.363877e+09
dload,0.0,2.242273e+07,2.412372e+06
dbytes,0.0,1.465753e+07,1.461993e+05
sbytes,24.0,1.435577e+07,1.737739e+05
response_body_len,0.0,6.558056e+06,4.962523e+04
sjit,0.0,1.483831e+06,4.903450e+04
rate,0.0,1.000000e+06,1.603446e+05
djit,0.0,4.631992e+05,3.930153e+03


In [8]:
corr = num.corr().abs()
pairs = corr.where(np.triu(np.ones(corr.shape), 1).astype(bool)).stack()
pairs = pairs[pairs > 0.95].sort_values(ascending=False)
pairs

is_ftp_login    ct_ftp_cmd          0.998855
dbytes          dloss               0.996711
sbytes          sloss               0.995772
swin            dwin                0.980458
dpkts           dloss               0.979612
ct_srv_src      ct_srv_dst          0.979467
dpkts           dbytes              0.973445
spkts           sloss               0.971859
                sbytes              0.964393
ct_dst_ltm      ct_src_dport_ltm    0.961518
ct_dst_src_ltm  ct_srv_dst          0.960321
ct_srv_src      ct_dst_src_ltm      0.953952
dtype: float64

In [9]:
((num - num.mean()).abs() > 3 * num.std()).mean().mul(100).round(2).sort_values(ascending=False).head(10)

ct_src_dport_ltm    3.44
smean               3.36
ct_dst_ltm          3.33
ct_src_ltm          3.30
dload               3.20
dmean               2.89
sload               1.81
rate                1.65
dur                 1.58
ct_srv_src          1.56
dtype: float64

In [10]:
# report: instances with ANY feature beyond 3 sd, and the widest numeric range
any_outlier = ((num - num.mean()).abs() > 3 * num.std()).any(axis=1).mean() * 100
print(f"instances with any feature >3 sd: {any_outlier:.2f}%")
print(f"sload range: {num['sload'].max():.3e}")

instances with any feature >3 sd: 23.29%
sload range: 5.988e+09


In [11]:
# report Table IV: service absent by class, and which protocols carry a service
by_class = (df.assign(c=df["attack_cat"].map(dict(enumerate(CLASSES))))
              .groupby("c", observed=True)["service"]
              .apply(lambda s: s.isna().mean() * 100)
              .sort_values().round(1))
print(by_class.to_string(), "\n")

has_svc = df.groupby("proto", observed=True)["service"].apply(lambda s: s.notna().any())
print(f"service present: {df['service'].notna().mean() * 100:.2f}%")
print(f"protocols with a service: {list(has_svc[has_svc].index)}")
print(f"protocols with none:      {(~has_svc).sum()} of {len(has_svc)}")

c
Generic             1.6
Worms              14.9
Exploits           51.8
Normal             68.7
Analysis           78.9
Reconnaissance     84.2
DoS                84.7
Fuzzers            90.7
Backdoor           95.3
Shellcode         100.0 

service present: 45.15%
protocols with a service: ['tcp', 'udp']
protocols with none:      131 of 133


## Feature setup

In [12]:
# drop the second feature of each correlated pair
DROP_CORR = sorted({b for _, b in pairs.index})
DROP_CORR

['ct_dst_src_ltm',
 'ct_ftp_cmd',
 'ct_src_dport_ltm',
 'ct_srv_dst',
 'dbytes',
 'dloss',
 'dwin',
 'sbytes',
 'sloss']

In [13]:
y = df["attack_cat"]
X = df.drop(columns=["id", "attack_cat"])
if SAMPLE_N:
    X, _, y, _ = train_test_split(X, y, train_size=SAMPLE_N, stratify=y, random_state=SEED)
NUMERIC = [c for c in X.columns if c not in NOMINAL]
X.shape

(257673, 42)

In [14]:
# one-hot min_frequency: levels kept and resulting dimensions per threshold.
# 1% keeps the 5 protocols above 1% (tcp, udp, unas, arp, ospf) and collapses
# the other 128; 1.5% would drop arp and ospf, which are not rare.
rows = []
for t in (0.1, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0):
    d = {"thresh %": t}
    for col in NOMINAL:
        vc = df[col].value_counts(dropna=False)
        kept = (vc / len(df) * 100 >= t).sum()
        d[col] = kept + (1 if kept < len(vc) else 0)   # + infrequent column
    d["total"] = sum(d[c] for c in NOMINAL)
    rows.append(d)
pd.DataFrame(rows).set_index("thresh %")

,proto,service,state,total
thresh %,,,,
0.10,11,9,5,25
0.25,7,9,5,21
0.50,7,9,5,21
0.75,6,7,5,18
1.00,6,7,5,18
1.50,4,7,4,15
2.00,4,6,4,14
3.00,4,4,4,12


## Pipelines

Two separate pre-processing paths. The tree gets no scaling, no correlation
removal and no one-hot encoding, because none of those change a rank-based split.

In [13]:
knn = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), [c for c in NUMERIC if c not in DROP_CORR]),
        ("nom", OneHotEncoder(min_frequency=0.01,
                              handle_unknown="infrequent_if_exist"), NOMINAL),
    ])),
    ("clf", KNeighborsClassifier()),
])

tree = Pipeline([
    ("prep", ColumnTransformer([
        ("num", "passthrough", NUMERIC),
        ("nom", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1,
                               encoded_missing_value=-1), NOMINAL),
    ])),
    ("clf", DecisionTreeClassifier(random_state=SEED)),
])

GRIDS = {
    "kNN": (knn, {"clf__n_neighbors": [1, 5, 11, 25],
                  "clf__weights": ["uniform", "distance"]}),
    "Tree": (tree, {"clf__max_depth": [None, 10, 20],
                    "clf__min_samples_leaf": [1, 5, 20],
                    "clf__criterion": ["gini", "entropy"],
                    "clf__class_weight": [None, "balanced"]}),
}

## Nested cross-validation

Tuning runs inside the outer folds, so no test fold informs a hyperparameter.

In [14]:
# tuned on f1_macro; accuracy and *_weighted are skew-blind, reported for contrast only
SCORING = [
    "f1_macro", "balanced_accuracy",
    "precision_macro", "recall_macro",
    "f1_weighted", "accuracy",
]

outer = StratifiedKFold(5, shuffle=True, random_state=SEED)
inner = StratifiedKFold(3, shuffle=True, random_state=SEED)

results = {}
for name, (pipe, grid) in GRIDS.items():
    gs = GridSearchCV(pipe, grid, scoring="f1_macro", cv=inner, n_jobs=-1)
    results[name] = cross_validate(gs, X, y, cv=outer, scoring=SCORING,
                                   return_estimator=True)

In [15]:
pd.DataFrame({(name, m): [r[f"test_{m}"].mean(), r[f"test_{m}"].std()]
              for name, r in results.items() for m in SCORING},
             index=["mean", "sd"]).T.round(4)

mean      sd
kNN  f1_macro           0.4663  0.0057
     balanced_accuracy  0.4475  0.0049
     precision_macro    0.5248  0.0060
     recall_macro       0.4475  0.0049
     f1_weighted        0.7602  0.0015
     accuracy           0.7647  0.0017
Tree f1_macro           0.5791  0.0062
     balanced_accuracy  0.5638  0.0073
     precision_macro    0.6557  0.0075
     recall_macro       0.5638  0.0073
     f1_weighted        0.8053  0.0014
     accuracy           0.8102  0.0030

In [16]:
# accuracy of always predicting the majority class
majority = y.value_counts(normalize=True).max()
print(f"majority-class (Normal) accuracy baseline: {majority:.4f}")

majority-class (Normal) accuracy baseline: 0.3609


In [17]:
for name, r in results.items():
    print(name)
    for e in r["estimator"]:
        print("   ", e.best_params_)

kNN
    {'clf__n_neighbors': 5, 'clf__weights': 'distance'}
    {'clf__n_neighbors': 5, 'clf__weights': 'distance'}
    {'clf__n_neighbors': 5, 'clf__weights': 'distance'}
    {'clf__n_neighbors': 5, 'clf__weights': 'distance'}
    {'clf__n_neighbors': 5, 'clf__weights': 'distance'}
Tree
    {'clf__class_weight': None, 'clf__criterion': 'entropy', 'clf__max_depth': None, 'clf__min_samples_leaf': 5}
    {'clf__class_weight': None, 'clf__criterion': 'entropy', 'clf__max_depth': None, 'clf__min_samples_leaf': 1}
    {'clf__class_weight': None, 'clf__criterion': 'gini', 'clf__max_depth': None, 'clf__min_samples_leaf': 5}
    {'clf__class_weight': None, 'clf__criterion': 'entropy', 'clf__max_depth': None, 'clf__min_samples_leaf': 5}
    {'clf__class_weight': None, 'clf__criterion': 'entropy', 'clf__max_depth': 20, 'clf__min_samples_leaf': 1}


## Confusion matrices

In [18]:
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)
labels = range(len(CLASSES))

for name, (pipe, _) in GRIDS.items():
    best = results[name]["estimator"][0].best_params_  # fold-0 params
    pred = pipe.set_params(**best).fit(Xtr, ytr).predict(Xte)
    print(name, best)
    print(pd.DataFrame(confusion_matrix(yte, pred, labels=labels),
                       index=CLASSES, columns=CLASSES))
    print(classification_report(yte, pred, labels=labels,
                                target_names=CLASSES, zero_division=0))

kNN {'clf__n_neighbors': 5, 'clf__weights': 'distance'}
                Normal  Reconnaissance  Backdoor   DoS  Exploits  Analysis  \
Normal           16732             214         1    22       184        13   
Reconnaissance     301            1615         6   208       499         0   
Backdoor            19              24         7   189       178         6   
DoS                119             155        46  1279      1542        11   
Exploits           444             382        63  1712      5956        18   
Analysis            43              12        13   191       192        50   
Fuzzers           1731             183        53   218       407        14   
Worms                3               3         0     0        19         0   
Shellcode           74              76         0     4        17         0   
Generic             23              13         1    55       151         0   

                Fuzzers  Worms  Shellcode  Generic  
Normal             1409      0  

## Paired comparison

Per-fold scores and the paired *t*-test quoted in the report.

In [19]:
from scipy import stats

folds = pd.DataFrame({(n, m): r[f"test_{m}"]
                      for n, r in results.items()
                      for m in ("f1_macro", "balanced_accuracy")})
print(folds.round(4).to_string(), "\n")

for m in ("f1_macro", "balanced_accuracy"):
    a, b = folds[("kNN", m)].values, folds[("Tree", m)].values
    d = b - a
    t, pv = stats.ttest_rel(b, a)
    # Nadeau-Bengio: folds share training data, so inflate the variance
    t_nb = d.mean() / np.sqrt((1 / len(d) + 1 / (len(d) - 1)) * d.var(ddof=1))
    p_nb = 2 * stats.t.sf(abs(t_nb), len(d) - 1)
    print(f"{m}: diff {d.mean():.4f}, tree wins {(d > 0).sum()}/5")
    print(f"   paired t={t:.2f} p={pv:.2e} | Nadeau-Bengio t={t_nb:.2f} p={p_nb:.2e}")

       kNN                       Tree                  
  f1_macro balanced_accuracy f1_macro balanced_accuracy
0   0.4701            0.4503   0.5850            0.5687
1   0.4618            0.4438   0.5820            0.5654
2   0.4658            0.4479   0.5696            0.5511
3   0.4589            0.4407   0.5848            0.5722
4   0.4750            0.4549   0.5741            0.5614 

f1_macro: diff 0.1128, tree wins 5/5
   paired t=22.56 p=2.29e-05 | Nadeau-Bengio t=15.04 p=1.14e-04
balanced_accuracy: diff 0.1162, tree wins 5/5
   paired t=22.52 p=2.30e-05 | Nadeau-Bengio t=15.01 p=1.15e-04
